# 02 — RF reconstruction and order classification

Reconstructs spatial receptive fields for all neurons across 17 containers and fits Gaussian-derivative models (orders m=0, 1, 2). Produces the canonical 1,017-neuron population table in `derived_data/population/`.

In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

population_dir  = REPO_ROOT / 'derived_data' / 'population'
review_dir      = REPO_ROOT / 'derived_data' / 'review_judgements'
cache_dir       = REPO_ROOT / 'data' / 'cache'
lists_dir       = REPO_ROOT / 'data' / 'experiment_lists'
population_dir.mkdir(parents=True, exist_ok=True)
import sys, warnings, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr, spearmanr, circstd, mannwhitneyu, linregress
from scipy.spatial.distance import pdist, squareform
from sklearn.linear_model import RidgeCV, Ridge
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

from rf_analysis.sparse_noise import (
    _build_design_matrix,
    _pixel_size,
    _LAMBDA_GRID,
    gaussian_deriv_m0,
    gaussian_deriv_m1,
    gaussian_deriv_m2,
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
warnings.filterwarnings('ignore')

In [ ]:
def batch_get_rf_maps_safe(dataset, requested_ids,
                            n_lags=8, response_delay=4, response_window=5):
    from rf_analysis.sparse_noise import _build_design_matrix, _pixel_size, _LAMBDA_GRID
    exp_cells = list(dataset.get_cell_specimen_ids())
    req_set   = set(int(c) for c in requested_ids)
    valid_ids = [c for c in exp_cells if c in req_set]
    if not valid_ids:
        raise ValueError('No requested cell_ids found in experiment.')

    stim_name = next((s for s in dataset.list_stimuli() if 'sparse_noise' in s), None)
    if stim_name is None:
        raise ValueError('No sparse noise stimulus.')

    stim_table = dataset.get_stimulus_table(stim_name)
    tr         = dataset.get_locally_sparse_noise_stimulus_template(stimulus=stim_name)
    template   = tr[0] if isinstance(tr, tuple) else tr
    n_tf, grid_h, grid_w = template.shape
    on_val, off_val = int(template.max()), int(template.min())
    pix_size        = _pixel_size(stim_name)

    starts        = stim_table['start'].values.astype(int)
    frame_indices = stim_table['frame'].values.astype(int)

    _, all_dff    = dataset.get_dff_traces(cell_specimen_ids=valid_ids)
    n_cells, n_tp = all_dff.shape

    X = _build_design_matrix(template, frame_indices, on_val, off_val,
                              grid_h, grid_w, n_lags)

    win_idx  = (starts + response_delay)[:, None] + np.arange(response_window)
    in_bnds  = (win_idx >= 0) & (win_idx < n_tp)
    win_safe = np.clip(win_idx, 0, n_tp - 1)
    dff_wins = all_dff[:, win_safe]
    dff_wins[:, ~in_bnds] = np.nan
    Y = np.nanmean(dff_wins, axis=2).T
    Y = np.where(np.isnan(Y), 0.0, Y).astype(np.float64)

    frame_valid  = (frame_indices >= 0) & (frame_indices < n_tf)
    X_fit, Y_fit = X[frame_valid].astype(np.float64), Y[frame_valid]
    if X_fit.shape[0] < 20:
        raise ValueError('Too few valid presentations.')

    try:
        rcv = RidgeCV(alphas=_LAMBDA_GRID, cv=None,
                      fit_intercept=True, alpha_per_target=True)
        rcv.fit(X_fit, Y_fit)
        W = np.atleast_2d(rcv.coef_)
    except TypeError:
        rcv = RidgeCV(alphas=_LAMBDA_GRID, cv=None, fit_intercept=True)
        rcv.fit(X_fit, Y_fit.mean(axis=1, keepdims=True))
        r = Ridge(alpha=float(rcv.alpha_), fit_intercept=True)
        r.fit(X_fit, Y_fit)
        W = np.atleast_2d(r.coef_)

    rf_out = {}
    for i, cid in enumerate(valid_ids):
        strf     = W[i].reshape(n_lags, grid_h, grid_w)
        best_lag = int(np.argmax([np.abs(strf[l]).max() for l in range(n_lags)]))
        rf_out[cid] = (strf[best_lag], pix_size)
    return rf_out

In [ ]:
QUALITY_THRESHOLD = 0.3
PIXEL_SIZE_UM     = 0.78

t_start = time.time()

for row_i, exp_row in session_c.iterrows():
    container_id = exp_row['experiment_container_id']
    exp_id       = exp_row['id']
    cre_line     = exp_row['cre_line']
    depth        = exp_row['imaging_depth']

    out_file = population_dir / f'rf_params_order_v2_container_{container_id}.csv'
    if out_file.exists():
        df_ex = pd.read_csv(out_file)
        continue

    t0 = time.time()

    try:
        dataset  = boc.get_ophys_experiment_data(exp_id)
        cell_ids = list(dataset.get_cell_specimen_ids())

        rf_cache = batch_get_rf_maps_safe(dataset, cell_ids)
    except Exception as e:
        continue

    nb05b_file = ridge_dir / f'rf_params_ridge_container_{container_id}.csv'
    cortex_lookup = {}
    if nb05b_file.exists():
        df_nb05b = pd.read_csv(nb05b_file)
        for _, r in df_nb05b.iterrows():
            cortex_lookup[int(r['cell_id'])] = (
                r.get('cortex_x_um', np.nan),
                r.get('cortex_y_um', np.nan),
                r.get('x0', np.nan),
                r.get('y0', np.nan),
            )

    rows = []
    for cid in cell_ids:
        if cid not in rf_cache:
            continue
        rf_map, pix = rf_cache[cid]
        rf_on  = np.maximum(rf_map,  0.0)
        rf_off = np.maximum(-rf_map, 0.0)

        try:
            best_m, best_p, best_q, all_res = classify_rf_order(
            rf_on, rf_off,
            pixel_size_deg=pix,
            delta_r2_threshold=0.0,
            n_random_starts=15,
            seed=cid,
            )

            if best_q['r_squared'] < QUALITY_THRESHOLD:
                continue

            cx, cy, x0, y0 = cortex_lookup.get(cid, (np.nan, np.nan, np.nan, np.nan))

            rows.append({
                'cell_id':       cid,
                'container_id':  container_id,
                'cre_line':      cre_line,
                'depth_um':      depth,
                'derivative_order': best_m,
                'sigma':         best_p['sigma'],
                'theta':         best_p['theta'],
                'kappa':         best_p['kappa'],
                'x0':            float(x0),
                'y0':            float(y0),
                'sigma_x':       best_p['sigma_x'],
                'sigma_y':       best_p['sigma_y'],
                'amplitude':     best_p['amplitude'],
                'dominant_subfield': best_p['dominant_subfield'],
                'r_squared':     best_q['r_squared'],
                'rmse':          best_q['rmse'],
                'aic':           best_q['aic'],
                'r2_m0':         all_res[0]['quality']['r_squared'],
                'r2_m1':         all_res[1]['quality']['r_squared'],
                'r2_m2':         all_res[2]['quality']['r_squared'],
                'delta_r2_vs_m0':     best_p.get('delta_r2_vs_m0', np.nan),
                'r2_improvement':     best_p.get('r2_improvement', np.nan),
                'n_starts_succeeded': best_p.get('n_starts_succeeded', np.nan),
                'cortex_x_um':   float(cx),
                'cortex_y_um':   float(cy),
                'phi':           best_p['phi'],
                'phi_confidence': best_p['phi_confidence'],
                'theta_hybrid':  best_p['theta_hybrid'],
                'method':        'order_class',
                
            })
        except Exception as e:
            warnings.warn(f'Cell {cid}: {e}')

    if rows:
        pd.DataFrame(rows).to_csv(out_file, index=False)
    else:
        pass


In [ ]:
csv_files = sorted(population_dir.glob('rf_params_order_v2_container_*.csv'))

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    if 'container_id' not in df.columns:
        df['container_id'] = int(f.stem.split('_')[-1])
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
for col in ['sigma', 'theta', 'kappa', 'r_squared', 'rmse', 'aic',
            'sigma_x', 'sigma_y', 'x0', 'y0', 'phi', 'phi_confidence', 'theta_hybrid',
            'cortex_x_um', 'cortex_y_um', 'r2_m0', 'r2_m1', 'r2_m2']:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors='coerce')

all_df['sigma_major']  = all_df[['sigma_x', 'sigma_y']].max(axis=1)
all_df['sigma_minor']  = all_df[['sigma_x', 'sigma_y']].min(axis=1)
all_df['log_sigma']    = np.log10(all_df['sigma'])
all_df['log_kappa']    = np.log(all_df['kappa'])

m0_mask = all_df['derivative_order'] == 0
all_df.loc[m0_mask, 'theta_hybrid'] = all_df.loc[m0_mask, 'theta']
m2_mask = all_df['derivative_order'] == 2
all_df.loc[m2_mask, 'theta_hybrid'] = all_df.loc[m2_mask, 'theta']

nb05b_files = sorted(ridge_dir.glob('rf_params_ridge_container_*.csv'))
nb05b_dfs = []
for f in nb05b_files:
    df = pd.read_csv(f)
    if 'container_id' not in df.columns:
        df['container_id'] = int(f.stem.split('_')[-1])
    nb05b_dfs.append(df)
nb05b_df = pd.concat(nb05b_dfs, ignore_index=True)
for col in ['sigma', 'theta', 'kappa', 'r_squared']:
    nb05b_df[col] = pd.to_numeric(nb05b_df[col], errors='coerce')

In [ ]:
judged      = {}
nb07_m1_ids = set()
nb07_path   = review_dir / 'manual_m_judgements.csv'
if nb07_path.exists():
    jdf = pd.read_csv(nb07_path)
    jdf = jdf[jdf['m_manual'] >= 0]
    for _, r in jdf.iterrows():
        judged[int(r['cell_id'])] = int(r['m_manual'])
        if int(r['m_manual']) == 1:
            nb07_m1_ids.add(int(r['cell_id']))
else:
    pass

later_sources = [
    (review_dir  / 'nb09_review_judgements.csv',       'nb09',          'm_manual'),
    (population_dir / 'targeted_review_judgements.csv',   'targeted',      'm_manual'),
    (population_dir / 'unreviewed_m1_judgements.csv',     'unreviewed_m1', 'm_manual'),
    (population_dir / 'rescue_review_judgements.csv',     'rescue',        'm_final'),
]
for path, label, col in later_sources:
    if not path.exists():
        continue
    jdf     = pd.read_csv(path)
    jdf     = jdf[jdf[col] >= 0]
    applied = skipped = 0
    for _, r in jdf.iterrows():
        cid   = int(r['cell_id'])
        m_new = int(r[col])
        if cid in nb07_m1_ids and m_new == 0:
            skipped += 1
            continue
        judged[cid] = m_new
        applied += 1

n_m1_judged = sum(1 for v in judged.values() if v == 1)

all_df['m_manual']          = all_df['cell_id'].map(judged)
all_df['manually_verified'] = all_df['cell_id'].isin(judged)
all_df['m_final'] = np.where(
    all_df['m_manual'].notna(),
    all_df['m_manual'],
    all_df['derivative_order']
).astype(int)

if 'delta_r2_vs_m0' in all_df.columns:
    demote = (
        (all_df['m_final'] == 2) &
        (~all_df['manually_verified']) &
        (all_df['delta_r2_vs_m0'] < 0.10)
    )
    all_df.loc[demote, 'm_final'] = 0

m1   = all_df[all_df['m_final'] == 1].copy()
m1m2 = m1.copy()

m1['confidence'] = pd.cut(
    m1['r_squared'],
    bins=[0, 0.35, 0.50, 1.0],
    labels=['low', 'medium', 'high']
)
m1m2 = m1.copy()

In [ ]:
from scipy.stats import pearsonr
M1_COLOR   = '#2196F3'
ORDER_COLORS = {1: M1_COLOR}

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

ax = fig.add_subplot(gs[0, 0])
counts = all_df['m_final'].value_counts().sort_index()
bar_colors = {0: '#4CAF50', 1: '#2196F3', 2: '#FF9800'}
ax.bar([f'm={m}' for m in counts.index], counts.values,
       color=[bar_colors[m] for m in counts.index],
       edgecolor='white', width=0.6)
for i, (m, c) in enumerate(counts.items()):
    ax.text(i, c + 5, f'{100*c/len(all_df):.0f}%', ha='center', fontsize=9)
ax.set_ylabel('Count')
ax.set_title(f'Derivative order\n(n={len(all_df)} total, m=2 n=2 excluded from figures)')

ax = fig.add_subplot(gs[0, 1])
bins = np.linspace(0, 30, 35)
ax.hist(m1['sigma'].dropna(), bins=bins, alpha=0.75, color=M1_COLOR,
        label=f'm=1  μ={m1["sigma"].mean():.1f}°  n={len(m1)}')
ax.axvline(m1['sigma'].mean(), color=M1_COLOR, ls='--', lw=1.5)
ax.set_xlabel('σ (degrees)')
ax.set_ylabel('Count')
sigma_cv = m1['sigma'].std() / m1['sigma'].mean()
ax.set_title(f'σ distribution  m=1\nCV={sigma_cv:.3f}')
ax.legend(fontsize=9)

ax = fig.add_subplot(gs[0, 2])
ax.hist(m1['kappa'].dropna(), bins=np.linspace(1, 10, 35), alpha=0.75, color=M1_COLOR,
        label=f'm=1  med={m1["kappa"].median():.2f}')
ax.set_xlabel('κ (elongation)')
ax.set_ylabel('Count')
ax.set_title('κ distribution  m=1')
ax.legend(fontsize=9)

ax = fig.add_subplot(gs[0, 3])
bins_t = np.linspace(0, 180, 19)
ax.hist(m1['theta_hybrid'].dropna(), bins=bins_t, alpha=0.75, color=M1_COLOR,
        label=f'm=1  n={len(m1)}')
ax.axhline(len(m1) / 18, color='k', ls='--', lw=1.5, label='Uniform')
ax.set_xlabel('θ_hybrid (°)')
ax.set_ylabel('Count')
ax.set_title('Orientation θ_hybrid\n(φ lobe geometry for m=1)')
ax.set_xlim(0, 180)
ax.legend(fontsize=9)

ax = fig.add_subplot(gs[1, 0])
hi_r2 = m1[m1['r_squared'] >= 0.5]
lo_r2 = m1[m1['r_squared'] <  0.5]
ax.hist(lo_r2['sigma'].dropna(), bins=25, alpha=0.6, color='#90CAF9',
        label=f'R²<0.5  n={len(lo_r2)}')
ax.hist(hi_r2['sigma'].dropna(), bins=25, alpha=0.75, color=M1_COLOR,
        label=f'R²≥0.5  n={len(hi_r2)}')
ax.set_xlabel('σ (degrees)')
ax.set_ylabel('Count')
ax.set_title('σ by R² confidence tier\nm=1')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, 1])
ax.hist(lo_r2['kappa'].dropna(), bins=25, alpha=0.6, color='#90CAF9',
        label=f'R²<0.5  n={len(lo_r2)}')
ax.hist(hi_r2['kappa'].dropna(), bins=25, alpha=0.75, color=M1_COLOR,
        label=f'R²≥0.5  n={len(hi_r2)}')
ax.set_xlabel('κ (elongation)')
ax.set_ylabel('Count')
ax.set_title('κ by R² confidence tier\nm=1')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, 2])
ax.hist(m1['r_squared'].dropna(), bins=20, range=(0, 1), alpha=0.75, color=M1_COLOR,
        label=f'μ={m1["r_squared"].mean():.3f}')
ax.axvline(0.5, color='red', ls='--', lw=1.5, label='R²=0.5')
ax.set_xlabel('R²')
ax.set_ylabel('Count')
ax.set_title('Fit quality R²  m=1')
ax.legend(fontsize=9)

ax = fig.add_subplot(gs[1, 3])
ax.scatter(m1['sigma'], m1['kappa'], s=18, alpha=0.55, color=M1_COLOR)
r_sk, p_sk = pearsonr(m1['sigma'].dropna(),
                       m1.loc[m1['sigma'].notna(), 'kappa'])
ax.set_xlabel('σ (degrees)')
ax.set_ylabel('κ (elongation)')
ax.set_title(f'σ–κ coupling  m=1\nr={r_sk:.3f}  p={p_sk:.2e}')

fig.suptitle(f'm=1 edge-detector simple cells — Lindeberg parameters\n'
             f'(n={len(m1)}, manually reviewed; m=2 n=2 excluded)',
             fontsize=13, fontweight='bold')
plt.savefig(population_dir / 'fig_05d_v2_parameter_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
bins_t = np.linspace(0, 180, 19)
ax.hist(m1['theta_hybrid'].dropna(), bins=bins_t, alpha=0.75, color=M1_COLOR,
        label=f'm=1  n={len(m1)}')
ax.axhline(len(m1) / 18, color='k', ls='--', lw=1.5, label='Uniform')
ax.set_xlabel('θ_hybrid (°)')
ax.set_ylabel('Count')
ax.set_title('θ_hybrid: φ lobe geometry\nm=1 orientation distribution')
ax.set_xlim(0, 180)
ax.legend(fontsize=9)

ax = axes[1]
sub1 = m1[m1['phi'].notna() & m1['theta'].notna()].copy()
diff1 = np.abs(sub1['theta'] - sub1['phi'])
diff1 = np.minimum(diff1, 180 - diff1)
ax.scatter(sub1['theta'], sub1['phi'], s=14, alpha=0.55, color=M1_COLOR,
           label=f'n={len(sub1)}  mean|Δ|={diff1.mean():.1f}°')
ax.plot([0, 180], [0, 180], 'k--', lw=1, label='y=x')
ax.set_xlabel('θ envelope (°)')
ax.set_ylabel('φ lobe geometry (°)')
ax.set_title('θ vs φ  m=1\n(φ used as θ_hybrid — agreement validates estimator)')
ax.set_xlim(0, 180); ax.set_ylim(0, 180)
ax.legend(fontsize=9)

ax = axes[2]
conf = m1['phi_confidence'].dropna()
ax.hist(conf, bins=25, alpha=0.75, color=M1_COLOR,
        label=f'median={conf.median():.1f}°')
mean_sigma = m1['sigma'].mean()
ax.axvline(mean_sigma, color='k', ls='--', lw=1.5,
           label=f'mean σ={mean_sigma:.1f}°')
ax.set_xlabel('φ confidence (lobe separation °)')
ax.set_ylabel('Count')
ax.set_title('φ reliability  m=1\n(confidence > σ = well-separated lobes)')
ax.legend(fontsize=9)

plt.suptitle('Orientation estimators: m=1 edge detectors', fontweight='bold')
plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_orientation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
def container_stats_m1(df, label):
    return df.groupby('container_id').agg(
        n          = ('sigma', 'count'),
        sigma_mean = ('sigma', 'mean'),
        sigma_std  = ('sigma', 'std'),
        kappa_med  = ('kappa', 'median'),
    ).assign(sigma_cv   = lambda x: x['sigma_std'] / x['sigma_mean'],
             cascade_ok = lambda x: x['sigma_std'] / x['sigma_mean'] < 0.3,
             source     = label).reset_index()

cs_m1 = container_stats_m1(m1, 'm=1')
sigma_cv_m1 = m1['sigma'].std() / m1['sigma'].mean()
cs_m1_filt  = cs_m1[cs_m1['n'] >= 3]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
colors = ['#2ecc71' if ok else '#e74c3c' for ok in cs_m1_filt['cascade_ok']]
ax.bar(range(len(cs_m1_filt)), cs_m1_filt['sigma_cv'],
       color=colors, edgecolor='none')
ax.axhline(0.3, color='black', ls='--', lw=1.5, label='CV=0.3 threshold')
ax.set_xticks(range(len(cs_m1_filt)))
ax.set_xticklabels([str(c)[-4:] for c in cs_m1_filt['container_id']],
                    rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Container (last 4 digits)')
ax.set_ylabel('CV of σ  (m=1 neurons)')
n_ok = cs_m1_filt['cascade_ok'].sum()
ax.set_title('m=1 simple cells')
ax.legend(fontsize=9)

ax = axes[1]
data_v = [m1[m1['container_id'] == cid]['sigma'].dropna().values
          for cid in cs_m1_filt['container_id']
          if len(m1[m1['container_id'] == cid]) >= 3]
labels_v = [str(cid)[-4:] for cid in cs_m1_filt['container_id']
            if len(m1[m1['container_id'] == cid]) >= 3]
if data_v:
    parts = ax.violinplot(data_v, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(M1_COLOR); pc.set_alpha(0.5)
    ax.set_xticks(range(1, len(labels_v) + 1))
    ax.set_xticklabels(labels_v, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Container')
ax.set_ylabel('σ (degrees)')
ax.set_title('σ per container  m=1')

plt.suptitle('Cascade Smoothing Hypothesis: CV(σ) — m=1 simple cells', fontweight='bold')
plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_cascade_smoothing.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
from scipy.stats import pearsonr, spearmanr

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
containers = m1['container_id'].unique()
cmap = plt.cm.tab20(np.linspace(0, 1, len(containers)))
for j, cid in enumerate(containers):
    sub = m1[m1['container_id'] == cid]
    ax.scatter(sub['sigma'], sub['kappa'], s=40, alpha=0.8,
               color=cmap[j], label=str(cid)[-4:])
r_sk, p_sk = pearsonr(m1['sigma'].dropna(),
                       m1.loc[m1['sigma'].notna(), 'kappa'])
r_sp, p_sp = spearmanr(m1['sigma'].dropna(),
                        m1.loc[m1['sigma'].notna(), 'kappa'])
ax.set_xlabel('σ (degrees)')
ax.set_ylabel('κ (elongation)')
ax.set_title(f'σ–κ coupling  m=1  (n={len(m1)})')
ax.legend(title='Container', fontsize=6, ncol=2,
          loc='upper right', markerscale=1.5)

ax = axes[1]
ax.hist(m1['sigma'].dropna(), bins=15, color=M1_COLOR, alpha=0.7,
        edgecolor='white')
for _, row in m1.iterrows():
    ax.axvline(row['sigma'], color=M1_COLOR, alpha=0.2, lw=0.8)
ax.axvline(m1['sigma'].mean(), color='black', lw=2, ls='--',
           label=f'mean={m1["sigma"].mean():.1f}°')
ax.set_xlabel('σ (degrees)')
ax.set_ylabel('Count')
ax.set_title(f'σ distribution  m=1\nCV={m1["sigma"].std()/m1["sigma"].mean():.3f}')
ax.legend(fontsize=9)

ax = axes[2]
ax.hist(m1['kappa'].dropna(), bins=15, color=M1_COLOR, alpha=0.7,
        edgecolor='white')
ax.axvline(m1['kappa'].median(), color='black', lw=2, ls='--',
           label=f'median={m1["kappa"].median():.2f}')
ax.set_xlabel('κ (elongation)')
ax.set_ylabel('Count')
ax.set_title(f'κ distribution  m=1')
ax.legend(fontsize=9)

plt.suptitle('Spatial scale and elongation — m=1 simple cells\n'
             '(spatial autocorrelogram not shown: n=31 across 13 containers, '
             '~2.4 neurons/container below analysis threshold)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_spatial_organisation.png',
            dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
BIN_DEG = 40.0
m1['zone'] = np.where(m1['x0'] < BIN_DEG, 'binocular', 'monocular')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(m1['x0'], m1['y0'], s=22, alpha=0.65, color=M1_COLOR, label='m=1')
ax.axvline(BIN_DEG, color='red', ls='--', lw=1.5, label=f'{BIN_DEG}° boundary')
ax.set_xlabel('RF centre azimuth x₀ (°)')
ax.set_ylabel('RF centre elevation y₀ (°)')
ax.set_title('RF centres  m=1 edge detectors')
ax.legend(fontsize=9)

bino = m1[m1['zone'] == 'binocular']
mono = m1[m1['zone'] == 'monocular']

ax = axes[1]
for data, label, col in [(bino, 'Binocular', '#2980b9'), (mono, 'Monocular', '#e74c3c')]:
    ax.hist(data['sigma'].dropna(), bins=18, alpha=0.65,
            label=f'{label} (n={len(data)})', color=col)
if len(bino) > 1 and len(mono) > 1:
    u, p = mannwhitneyu(bino['sigma'].dropna(), mono['sigma'].dropna(), alternative='two-sided')
    ax.set_title(f'σ by zone  m=1\nMW p={p:.3f}')
ax.set_xlabel('σ (degrees)')
ax.legend(fontsize=9)

ax = axes[2]
for data, label, col in [(bino, 'Binocular', '#2980b9'), (mono, 'Monocular', '#e74c3c')]:
    ax.hist(data['kappa'].dropna(), bins=18, alpha=0.65, label=label, color=col)
if len(bino) > 1 and len(mono) > 1:
    u, p = mannwhitneyu(bino['kappa'].dropna(), mono['kappa'].dropna(), alternative='two-sided')
    ax.set_title(f'κ by zone  m=1\nMW p={p:.3f}')
ax.set_xlabel('κ (elongation)')
ax.legend(fontsize=9)

plt.suptitle('Binocular vs Monocular zone — m=1 simple cells', fontweight='bold')
plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_binocular_monocular.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

ax = axes[0]
log_s = np.log10(m1['sigma'].dropna())
ax.hist(log_s, bins=25, alpha=0.75, color=M1_COLOR,
        label=f'm=1  μ={log_s.mean():.2f}')
ax.set_xlabel('log₁₀(σ)')
ax.set_ylabel('Count')
ax.set_title('Log scale σ  m=1')
ax.legend(fontsize=9)

ax = axes[1]
ax.scatter(m1['sigma'], m1['sigma_major'], s=10, alpha=0.5,
           color='#e74c3c', label='σ_major')
ax.scatter(m1['sigma'], m1['sigma_minor'], s=10, alpha=0.5,
           color='#3498db', label='σ_minor')
ax.plot([0, 25], [0, 25], 'k--', lw=1)
r_maj, _ = pearsonr(m1['sigma'].dropna(),
                     m1.loc[m1['sigma'].notna(), 'sigma_major'])
ax.set_xlabel('σ geometric mean (°)')
ax.set_ylabel('Axis σ (°)')
ax.set_title(f'Axis decomposition  m=1\nr(σ,major)={r_maj:.2f}')
ax.legend(fontsize=9)

ax = axes[2]
ax.hist(m1['log_kappa'].dropna(), bins=25, alpha=0.75, color=M1_COLOR,
        label=f'med={m1["log_kappa"].median():.2f}')
ax.set_xlabel('ln(κ)')
ax.set_ylabel('Count')
ax.set_title('ln(κ)  m=1')
ax.legend(fontsize=9)

ax = axes[3]
ax.scatter(m1['log_sigma'], m1['log_kappa'], s=10, alpha=0.55, color=M1_COLOR)
r_log, p_log = pearsonr(m1['log_sigma'].dropna(),
                         m1.loc[m1['log_sigma'].notna(), 'log_kappa'])
ax.set_xlabel('log₁₀(σ)')
ax.set_ylabel('ln(κ)')
ax.set_title(f'log σ–κ coupling  m=1\nr={r_log:.3f}  p={p_log:.2e}')

plt.suptitle('Log-scale distributions and elongation — m=1', fontweight='bold')
plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_logscale.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import circstd

def summarise(df, label):
    sigma_cv = df['sigma'].std() / df['sigma'].mean()
    th_rad2  = np.deg2rad(df['theta_hybrid'].dropna() * 2)
    circ_sd  = np.rad2deg(circstd(th_rad2) / 2)
    r_sk, p_sk = pearsonr(df['sigma'].dropna(),
                           df.loc[df['sigma'].notna(), 'kappa'])

summarise(m1, 'm=1 edge-detector simple cells (manually reviewed)')

hi = m1[m1['r_squared'] >= 0.5]
lo = m1[m1['r_squared'] <  0.5]


In [ ]:
out_all = population_dir / 'rf_params_order_v2_all_containers.csv'
out_m1  = population_dir / 'rf_params_order_v2_m1_final.csv'

all_df.to_csv(out_all, index=False)
m1.to_csv(out_m1, index=False)

In [ ]:
from scipy.stats import mannwhitneyu as _mwu

HORIZ_WINDOW = 20

df_chk = m1[['theta_hybrid', 'theta', 'kappa', 'r_squared']].dropna()

near_h_hyb = df_chk[(df_chk['theta_hybrid'] < HORIZ_WINDOW) |
                     (df_chk['theta_hybrid'] > (180 - HORIZ_WINDOW))]
other_hyb  = df_chk[(df_chk['theta_hybrid'] >= HORIZ_WINDOW) &
                     (df_chk['theta_hybrid'] <= (180 - HORIZ_WINDOW))]
near_h_env = df_chk[(df_chk['theta'] < HORIZ_WINDOW) |
                     (df_chk['theta'] > (180 - HORIZ_WINDOW))]
other_env  = df_chk[(df_chk['theta'] >= HORIZ_WINDOW) &
                     (df_chk['theta'] <= (180 - HORIZ_WINDOW))]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Cardinal orientation bias  m=1 edge detectors\n'
             'Row 1: θ envelope   Row 2: θ_hybrid (φ lobe geometry)',
             fontsize=11, fontweight='bold')

bins = np.linspace(0, 180, 19)
kappa_med = df_chk['kappa'].median()

for row, (orient_col, near_h, other, label) in enumerate([
    ('theta',        near_h_env, other_env, 'θ envelope'),
    ('theta_hybrid', near_h_hyb, other_hyb, 'θ_hybrid (φ)'),
]):
    low_k  = df_chk[df_chk['kappa'] <  kappa_med]
    high_k = df_chk[df_chk['kappa'] >= kappa_med]

    ax = axes[row, 0]
    ax.hist(low_k[orient_col],  bins=bins, alpha=0.6, density=True,
            label=f'κ<{kappa_med:.2f}  n={len(low_k)}')
    ax.hist(high_k[orient_col], bins=bins, alpha=0.6, density=True,
            label=f'κ≥{kappa_med:.2f}  n={len(high_k)}')
    ax.axhline(1/180, color='k', ls='--', lw=1, label='Uniform')
    ax.set_xlabel(f'{orient_col} (°)')
    ax.set_ylabel('Density')
    ax.set_title(f'{label}\nθ split by κ')
    ax.set_xlim(0, 180)
    ax.legend(fontsize=7)

    ax = axes[row, 1]
    u, p = _mwu(near_h['kappa'], other['kappa'], alternative='two-sided')
    ax.hist(near_h['kappa'], bins=20, alpha=0.65,
            label=f'Near-horiz n={len(near_h)}  med={near_h["kappa"].median():.2f}')
    ax.hist(other['kappa'],  bins=20, alpha=0.65,
            label=f'Other n={len(other)}  med={other["kappa"].median():.2f}')
    ax.set_xlabel('κ')
    ax.set_ylabel('Count')
    ax.set_title(f'{label}\nκ near-horiz vs other  MW p={p:.4f}')
    ax.legend(fontsize=7)

    ax = axes[row, 2]
    u2, p2 = _mwu(near_h['r_squared'], other['r_squared'], alternative='two-sided')
    ax.hist(near_h['r_squared'], bins=15, range=(0, 1), alpha=0.65,
            label=f'Near-horiz  med={near_h["r_squared"].median():.3f}')
    ax.hist(other['r_squared'],  bins=15, range=(0, 1), alpha=0.65,
            label=f'Other  med={other["r_squared"].median():.3f}')
    ax.set_xlabel('R²')
    ax.set_ylabel('Count')
    ax.set_title(f'{label}\nR² near-horiz vs other  MW p={p2:.4f}')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_cardinal_bias.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
from scipy.stats import circstd

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
bins_t = np.linspace(0, 180, 13)
counts_hyb, _ = np.histogram(m1['theta_hybrid'].dropna(), bins=bins_t)
counts_env, _ = np.histogram(m1['theta'].dropna(),         bins=bins_t)
expected = m1['theta_hybrid'].notna().sum() / 12

ax.bar(bins_t[:-1], counts_env,  width=14, alpha=0.4, color='grey',
       label='θ envelope', align='edge')
ax.bar(bins_t[:-1], counts_hyb, width=14, alpha=0.65, color=M1_COLOR,
       label='θ_hybrid (φ)', align='edge')
ax.axhline(expected, color='black', ls='--', lw=1.5, label='Uniform')
chi2_hyb = float(np.sum((counts_hyb - expected)**2 / expected))
chi2_env = float(np.sum((counts_env - expected)**2 / expected))
ax.set_xlabel('Orientation (°)')
ax.set_ylabel('Count')
ax.set_title(f'm=1 orientation distribution\nχ²(φ)={chi2_hyb:.1f}  χ²(env)={chi2_env:.1f}  df=11')
ax.set_xlim(0, 180)
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(m1['sigma'], m1['theta_hybrid'], s=12, alpha=0.5, color=M1_COLOR)
r_st, p_st = pearsonr(m1['sigma'].dropna(),
                       m1.loc[m1['sigma'].notna(), 'theta_hybrid'])
ax.set_xlabel('σ (°)')
ax.set_ylabel('θ_hybrid (°)')
ax.set_title(f'θ_hybrid vs σ  m=1\nr={r_st:.3f}  p={p_st:.3f}')
ax.set_ylim(0, 180)

ax = axes[2]
ax.scatter(m1['kappa'], m1['theta_hybrid'], s=12, alpha=0.5, color=M1_COLOR)
r_kt, p_kt = pearsonr(m1['kappa'].dropna(),
                       m1.loc[m1['kappa'].notna(), 'theta_hybrid'])
ax.set_xlabel('κ (elongation)')
ax.set_ylabel('θ_hybrid (°)')
ax.set_title(f'θ_hybrid vs κ  m=1\nr={r_kt:.3f}  p={p_kt:.3f}')
ax.set_ylim(0, 180)

plt.suptitle('θ_hybrid uniformity and parameter correlations  m=1', fontweight='bold')
plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_theta_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

csd = np.rad2deg(circstd(np.deg2rad(m1['theta_hybrid'].dropna() * 2)) / 2)
csd_env = np.rad2deg(circstd(np.deg2rad(m1['theta'].dropna() * 2)) / 2)

In [ ]:
from scipy.stats import circstd

fig = plt.figure(figsize=(10, 7))
panels = [
    ('theta',        'θ envelope',     '#888888'),
    ('theta_hybrid', 'θ_hybrid (φ)',   M1_COLOR),
]
bins_polar = np.linspace(0, 2*np.pi, 37)
bin_centres = 0.5 * (bins_polar[:-1] + bins_polar[1:])
width_p     = bins_polar[1] - bins_polar[0]

for col_idx, (col, label, color) in enumerate(panels):
    ax = fig.add_subplot(1, 2, col_idx + 1, projection='polar')
    data = m1[col].dropna()
    angles = np.deg2rad(data.values * 2)
    counts, _ = np.histogram(angles, bins=bins_polar)
    ax.bar(bin_centres, counts, width=width_p, color=color,
           alpha=0.75, edgecolor='white', linewidth=0.3)
    uniform_r = len(data) / 36
    ax.plot(np.linspace(0, 2*np.pi, 100), [uniform_r]*100,
            'k--', lw=0.8, alpha=0.5)
    csd = np.rad2deg(circstd(np.deg2rad(data * 2)) / 2)
    near_h = ((data < 20) | (data > 160)).mean() * 100
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)
    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['0°','','90°','','180°','','270°',''], fontsize=7)
    ax.set_yticklabels([])
    ax.set_title(f'm=1  {label}\nn={len(data)}  SD={csd:.1f}°  horiz={near_h:.0f}%',
                 fontsize=10, color=color, pad=15)

fig.suptitle('Polar rose: m=1 orientation distribution\n'
             'θ envelope vs θ_hybrid (φ lobe geometry)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(population_dir / 'fig_05d_v2_orientation_polar.png', dpi=150, bbox_inches='tight')
plt.show()